In [23]:
import os
import json
import pandas as pd

import data_parser as dp

In [25]:
# Pfad zum aktuellen Verzeichnis
current_dir = os.getcwd()

# Pfad zum übergeordneten Verzeichnis
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
print(current_dir)
print(parent_dir)

/Users/asuka/Desktop/DATA_parser
/Users/asuka/Desktop


### Sleep_data

In [ ]:
filepath = parent_dir + '/data/Liza/sleep_data.json'
sleep_data = dp.load_json_data(filepath)

print(type(sleep_data))

In [ ]:
df_sleep_L = pd.DataFrame(dp.parse_sleep(sleep_data))
df_sleep_L = df_sleep_L.dropna(axis=0)
df_sleep_L

### Stress_data

In [ ]:
filepath = parent_dir + '/data/Liza/stress_data.json'
stress_data = dp.load_json_data(filepath)

print(type(stress_data))

In [ ]:
df_stress_L = pd.DataFrame(dp.parse_stress(stress_data))
df_stress_L = df_stress_L.dropna(axis=0)
df_stress_L

### Heart_rates

In [ ]:
filepath = parent_dir + '/data/Liza/heart_rates.json'
heart_rates = dp.load_json_data(filepath)

print(type(heart_rates))

In [ ]:
df_heart_rates_L = pd.DataFrame(dp.parse_heart_rates(heart_rates))
df_heart_rates_L = df_heart_rates_L.dropna(axis=0)
df_heart_rates_L

### Body_battery

In [ ]:
filepath = parent_dir + '/data/Liza/body_battery.json'
body_battery_data = dp.load_json_data(filepath)

print(type(body_battery_data))

In [ ]:
df_body_battery_L = pd.DataFrame(dp.parse_body_battery(body_battery_data))
df_body_battery_L = df_body_battery_L.dropna(axis=0)
df_body_battery_L

### Iter files

In [3]:
current_dir = os.getcwd()
stress_data, heart_rates, sleep_data, body_battery = dp.iter_files(os.path.join(parent_dir, "data"))

# DataFrames erstellen
df_stress_data = pd.DataFrame(stress_data)
df_heart_rates = pd.DataFrame(heart_rates)
df_sleep_data = pd.DataFrame(sleep_data)
df_body_battery = pd.DataFrame(body_battery)

## Cleaning data

In [ ]:
#df_sleep = df_sleep_data.dropna(axis=0)
#df_stress = df_stress_data.dropna(axis=0)
#df_heart_rates = df_heart_rates.dropna(axis=0)
#df_body_battery = df_body_battery.dropna(axis=0)

In [4]:
print(f"sleep: {len(df_sleep_data)}")
print(f"stress: {len(df_stress_data)}")
print(f"heart_rates: {len(df_heart_rates)}")
print(f"body_battery: {len(df_body_battery)}")

sleep: 375
stress: 375
heart_rates: 375
body_battery: 375


## Random Forest

In [5]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report

In [6]:
# Setze die Optionen, um alle Zeilen und Spalten anzuzeigen
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [7]:
# Zusammenführen der DataFrames anhand des Datums
df_merged = df_body_battery.merge(df_heart_rates, left_on='date', right_on='calendarDate', how='outer')
df_merged = df_merged.merge(df_stress_data, on='calendarDate', how='outer')
df_merged = df_merged.merge(df_sleep_data, on='calendarDate', how='outer')

In [8]:
df_merged = df_merged.dropna(axis=0)
print(len(df_merged))

7755


In [10]:
# Spaltennamen ausgeben
print(df_merged.columns)

Index(['date', 'charged', 'drained', 'startTimestampGMT', 'endTimestampGMT',
       'startTimestampLocal', 'endTimestampLocal', 'endOfDayBodyBatteryLevel',
       'calendarDate', 'maxHeartRate', 'minHeartRate', 'restingHeartRate',
       'lastSevenDaysAvgRestingHeartRate', 'maxStressLevel', 'avgStressLevel',
       'sleepTimeSeconds', 'deepSleepSeconds', 'lightSleepSeconds',
       'remSleepSeconds', 'awakeSleepSeconds', 'sleepQuality'],
      dtype='object')


In [11]:
df_merged.iloc[0]

date                                           2023-12-07
charged                                              64.0
drained                                              68.0
startTimestampGMT                   2023-12-06T23:00:00.0
endTimestampGMT                     2023-12-07T23:00:00.0
startTimestampLocal                 2023-12-07T00:00:00.0
endTimestampLocal                   2023-12-08T00:00:00.0
endOfDayBodyBatteryLevel                             HIGH
calendarDate                                   2023-12-07
maxHeartRate                                        150.0
minHeartRate                                         66.0
restingHeartRate                                     85.0
lastSevenDaysAvgRestingHeartRate                     85.0
maxStressLevel                                       95.0
avgStressLevel                                       41.0
sleepTimeSeconds                                  24060.0
deepSleepSeconds                                   4140.0
lightSleepSeco

In [12]:
df_merged['endOfDayBodyBatteryLevel'].unique()

array(['HIGH', 'MODERATE', 'LOW', 'VERY_LOW'], dtype=object)

In [13]:
# Beispiel für Feature-Auswahl
features = df_merged[[
    'charged', 'drained',
    'endOfDayBodyBatteryLevel',
    'maxHeartRate', 'minHeartRate', 'restingHeartRate',
    'lastSevenDaysAvgRestingHeartRate',
    'sleepTimeSeconds', 'deepSleepSeconds', 'lightSleepSeconds',
    'remSleepSeconds', 'awakeSleepSeconds', 'sleepQuality'
]]

### Feature-Transformation

In [ ]:

features['sleepEfficiency'] = features['sleepTimeSeconds'] / (24 * 3600)

In [14]:
# Die Zuordnung von den Werten zu den Zahlen definieren
mapping = {'HIGH': 3, 'MODERATE': 2, 'LOW': 1, 'VERY_LOW': 0}

# Die Spalte 'ColumnName' in deinem DataFrame umwandeln
features['endOfDayBodyBatteryLevel'] = features['endOfDayBodyBatteryLevel'].map(mapping)

/var/folders/3p/pzbsmt9j4c9bkdp8gb2bdq280000gn/T/ipykernel_11273/4086282735.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features['endOfDayBodyBatteryLevel'] = features['endOfDayBodyBatteryLevel'].map(mapping)


### Modell

In [15]:
from sklearn.model_selection import train_test_split

# Angenommen, 'df_merged['stressCause']' ist unsere Zielvariable
X = features
# Setzen eines Schwellenwerts für hohe vs. niedrige Stresslevel
threshold = df_merged['avgStressLevel'].median()
y = (df_merged['avgStressLevel'] > threshold).astype(int)  # 1 für hoch, 0 für niedrig

# Aufteilung der Daten in Trainings- und Testsets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [16]:
# Erstellen des Random Forest Klassifikators
random_forest_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

# Training des Modells mit den Trainingsdaten
random_forest_classifier.fit(X_train, y_train)


RandomForestClassifier(random_state=42)

In [17]:
# Erstellen des Random Forest Regressors
random_forest_regressor = RandomForestRegressor(n_estimators=100, random_state=42)

# Training des Modells mit den Trainingsdaten
random_forest_regressor.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [18]:
# Vorhersagen auf dem Testset
y_pred = random_forest_classifier.predict(X_test)

# Bewertung der Genauigkeit
accuracy = accuracy_score(y_test, y_pred)
print(f'Genauigkeit: {accuracy:.4f}')

Genauigkeit: 0.4887


In [19]:
# Vorhersagen auf dem Testset
y_pred = random_forest_regressor.predict(X_test)

# Bewertung des mittleren quadratischen Fehlers (MSE)
mse = mean_squared_error(y_test, y_pred)
print(f'Mittlerer quadratischer Fehler (MSE): {mse:.4f}')

Mittlerer quadratischer Fehler (MSE): 0.3321


In [20]:
# Erstellen des Random Forest Klassifikators
clf = RandomForestClassifier(n_estimators=100, random_state=42)

# Trainieren des Modells mit dem Trainingsset
clf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [21]:
# Vorhersagen mit dem Testset
y_pred = clf.predict(X_test)

# Bewertung der Modellleistung
accuracy = accuracy_score(y_test, y_pred)
print(f"Genauigkeit: {accuracy:.2f}")

# Ausführlicher Klassifikationsbericht
report = classification_report(y_test, y_pred)
print(report)


Genauigkeit: 0.49
              precision    recall  f1-score   support

           0       0.49      0.50      0.50       782
           1       0.48      0.47      0.48       769

    accuracy                           0.49      1551
   macro avg       0.49      0.49      0.49      1551
weighted avg       0.49      0.49      0.49      1551



In [22]:
importances = clf.feature_importances_
feature_names = X_train.columns
feature_importance_dict = {name: importance for name, importance in zip(feature_names, importances)}

# Sortierung der Merkmale nach ihrer Wichtigkeit
sorted_importances = sorted(feature_importance_dict.items(), key=lambda x: x[1], reverse=True)

print("Wichtigkeit der Merkmale:")
for feature, importance in sorted_importances:
    print(f"{feature}: {importance:.4f}")

Wichtigkeit der Merkmale:
charged: 0.1291
drained: 0.1097
maxHeartRate: 0.0998
restingHeartRate: 0.0859
minHeartRate: 0.0855
lastSevenDaysAvgRestingHeartRate: 0.0834
remSleepSeconds: 0.0722
sleepQuality: 0.0689
sleepTimeSeconds: 0.0654
lightSleepSeconds: 0.0646
awakeSleepSeconds: 0.0602
deepSleepSeconds: 0.0528
endOfDayBodyBatteryLevel: 0.0226
